# Data Cleaning & Standardization

Standardize potency values, deduplicate, handle censored values, and canonicalize SMILES for the raw KIT bioactivity pull from `01_data_collection.ipynb`.

See [Design Doc.md](../Design%20Doc.md) §5.1 and §4.4, and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 2.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_PATH = Path("../data/raw/chembl_kit_bioactivity_raw.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print("Loaded:", df.shape)
df.head()

Loaded: (8703, 18)


,assay_chembl_id,assay_description,assay_type,canonical_smiles,document_chembl_id,document_year,molecule_chembl_id,pchembl_value,relation,standard_relation,standard_type,standard_units,standard_value,target_chembl_id,target_organism,type,units,value
0,CHEMBL820421,Inhibition of c-Kit autophosphorylation in int...,B,COc1cc2c(Oc3ccc(Nc4ccc(C(C)(C)C)cc4)cc3)ccnc2c...,CHEMBL1146677,2004.0,CHEMBL352308,7.00,=,=,IC50,nM,100.0,CHEMBL1936,Homo sapiens,IC50,nM,100.000
1,CHEMBL702237,Inhibition of KIT kinase activity,B,O=C(Cc1ccc2ccccc2c1)Nc1cc(C2CC2)n[nH]1,CHEMBL1148336,2004.0,CHEMBL115220,NaN,>,>,IC50,nM,10000.0,CHEMBL1936,Homo sapiens,IC50,nM,10000.000
2,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc...,CHEMBL1135998,2002.0,CHEMBL330863,7.68,=,=,IC50,nM,21.0,CHEMBL1936,Homo sapiens,IC50,uM,0.021
3,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc...,CHEMBL1135998,2002.0,CHEMBL124660,6.77,=,=,IC50,nM,170.0,CHEMBL1936,Homo sapiens,IC50,uM,0.170
4,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(C#N)cc4)CC3)ncnc2cc...,CHEMBL1135998,2002.0,CHEMBL126699,8.22,=,=,IC50,nM,6.0,CHEMBL1936,Homo sapiens,IC50,uM,0.006


## 1. Standardize potency values to pIC50/pKi/pEC50/pKd

Design Doc §5.1: convert to a common scale, pX = -log10(molar concentration), so IC50/Ki/Kd/EC50 values become comparable and roughly normally distributed. `standard_units` is almost all `nM` (8,549/8,703 — see `01_data_collection.ipynb` null counts), but the converter below handles the full range of molar-concentration unit strings ChEMBL uses, for robustness.

This step converts every row that *has* a value and unit, regardless of `standard_relation` (`=`, `>`, `<`, etc.) — a censored value like ">10000 nM" still converts to a valid bound, "pIC50 < 4.0". Deciding what to *do* with censored rows (drop/cap/flag) is a separate, later step, so the relation is carried through unchanged here rather than being resolved now.

In [2]:
# Molar-concentration unit strings ChEMBL uses -> multiplier to convert to molar (M).
UNITS_TO_MOLAR = {
    "M": 1,
    "mM": 1e-3,
    "uM": 1e-6,
    "nM": 1e-9,
    "pM": 1e-12,
    "fM": 1e-15,
}

unknown_units = set(df["standard_units"].dropna().unique()) - set(UNITS_TO_MOLAR)
assert not unknown_units, f"Unhandled units present, extend UNITS_TO_MOLAR: {unknown_units}"

molar = df["standard_value"] * df["standard_units"].map(UNITS_TO_MOLAR)
df["p_value"] = -np.log10(molar)

n_missing = df["p_value"].isna().sum()
print(f"p_value computed for {df['p_value'].notna().sum()} / {len(df)} rows")
print(f"{n_missing} rows have no p_value (missing standard_value and/or standard_units)")
df[["standard_type", "standard_relation", "standard_value", "standard_units", "pchembl_value", "p_value"]].head(10)

p_value computed for 8547 / 8703 rows
156 rows have no p_value (missing standard_value and/or standard_units)


,standard_type,standard_relation,standard_value,standard_units,pchembl_value,p_value
0,IC50,=,100.0,nM,7.00,7.000000
1,IC50,>,10000.0,nM,NaN,5.000000
2,IC50,=,21.0,nM,7.68,7.677781
3,IC50,=,170.0,nM,6.77,6.769551
4,IC50,=,6.0,nM,8.22,8.221849
5,IC50,=,4.0,nM,8.40,8.397940
6,IC50,=,260.0,nM,6.58,6.585027
7,IC50,=,190.0,nM,6.72,6.721246
8,IC50,=,60.0,nM,7.22,7.221849
9,IC50,=,16.0,nM,7.80,7.795880


### Cross-check against ChEMBL's own `pchembl_value`

ChEMBL independently computes `pchembl_value` (its own -log10 molar normalization) for a subset of records — mostly `=`-relation, non-censored ones. Our `p_value` should match it closely; large discrepancies would indicate a bug in the unit conversion above.

In [3]:
both = df.dropna(subset=["pchembl_value", "p_value"])
diff = (both["p_value"] - both["pchembl_value"]).abs()

print(f"{len(both)} rows have both p_value and pchembl_value")
print("Max abs difference:", diff.max())
print("Rows with |diff| > 0.01:", (diff > 0.01).sum())

assert diff.max() < 0.01, "p_value disagrees with ChEMBL's own pchembl_value — check unit conversion"

5711 rows have both p_value and pchembl_value
Max abs difference: 0.005483746814912038
Rows with |diff| > 0.01: 0


## 2. Policy for mixed assay types (IC50 / Ki / Kd / EC50)

Design Doc §4.4 flags that IC50/Ki/Kd aren't directly comparable without care. Rather than assume a rule, this checks the actual data:

- **`assay_type` composition** (B=binding, F=functional): IC50, Kd, and EC50 are ~97-99% binding-type assays; Ki is 76% binding / 24% functional — the functional-Ki slice is the one non-standard chunk worth flagging.
- **Cross-type divergence for the same compound** (`=`-relation records only, median of per-compound-per-type values): IC50 vs Ki median |diff| = 0.80 log units (n=30 shared compounds), IC50 vs Kd = 0.62 (n=31), Ki vs Kd = 0.45 (n=9).

**Decision:** pool IC50 + Ki + Kd into a single regression target. The overlap between types is too small (~30 compounds) to fit a reliable cross-type calibration, and the observed offset (0.5-0.8 log units) is the same order of magnitude as the paper-to-paper noise already being tolerated in the dedup/aggregation step (§2.3 below) — so it's treated as part of the same measurement-noise budget rather than a separate correction problem. **EC50 is excluded**: only 206 records (~2% of the data) and its provenance is murkier (a mix of true functional readouts and inhibition curves some papers happen to label EC50), so dropping it costs little and removes an ambiguity. `standard_type` is kept as a column in the processed output so it remains available later as an optional model feature or for a type-restricted sensitivity check, rather than being silently discarded.

In [4]:
# --- assay_type composition per standard_type ---
print("=== assay_type (B=binding, F=functional) by standard_type ===")
print(pd.crosstab(df["standard_type"], df["assay_type"]))
print()

# --- cross-type divergence for compounds measured in more than one type ---
eq = df[(df["standard_relation"] == "=") & df["p_value"].notna()]
med_by_type = eq.groupby(["molecule_chembl_id", "standard_type"])["p_value"].median().unstack()

print("=== cross-type divergence (median of per-compound-per-type p_value) ===")
for t1, t2 in [("IC50", "Ki"), ("IC50", "Kd"), ("IC50", "EC50"), ("Ki", "Kd")]:
    both = med_by_type[[t1, t2]].dropna()
    if len(both) > 0:
        diff = both[t1] - both[t2]
        print(
            f"{t1} vs {t2}: n={len(both)}, mean diff={diff.mean():.2f}, "
            f"median |diff|={diff.abs().median():.2f}"
        )

=== assay_type (B=binding, F=functional) by standard_type ===
assay_type      A     B    F   T
standard_type                   
EC50            6   200    0   0
IC50           68  4354   43  20
Kd              1  1234    0   2
Ki              0  2108  667   0

=== cross-type divergence (median of per-compound-per-type p_value) ===
IC50 vs Ki: n=30, mean diff=-0.24, median |diff|=0.80
IC50 vs Kd: n=31, mean diff=-0.68, median |diff|=0.62
IC50 vs EC50: n=5, mean diff=0.54, median |diff|=0.67
Ki vs Kd: n=9, mean diff=-0.51, median |diff|=0.45


In [5]:
n_before = len(df)

INCLUDED_TYPES = ["IC50", "Ki", "Kd"]
df = df[df["standard_type"].isin(INCLUDED_TYPES)].reset_index(drop=True)

print(f"Rows before type filter: {n_before}")
print(f"Rows after dropping EC50: {len(df)} ({n_before - len(df)} removed)")
df["standard_type"].value_counts()

Rows before type filter: 8703
Rows after dropping EC50: 8497 (206 removed)


standard_type
IC50    4485
Ki      2775
Kd      1237
Name: count, dtype: int64

## 3. Deduplicate & aggregate repeated measurements

Design Doc §5.1 calls for deduplicating repeated measurements per compound via median of log-transformed values. Doing that naively per `molecule_chembl_id` would be wrong here: **746 compounds have both wild-type and D816V-mutant measurements, and 226 of those (30%) differ by more than 1 log unit (10x)** between the two — that's exactly the mutant-selectivity signal the Phase 5 analysis is built to detect. Averaging WT and D816V together per compound would destroy it before it ever reaches that notebook.

So the wild-type/D816V-mutant flag (planned as a later step in [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 2) is computed here instead, and aggregation groups by **`(molecule_chembl_id, kit_variant)`**, not `molecule_chembl_id` alone. Same local substring search validated in `01_data_collection.ipynb`: `assay_description` mentioning "D816V" -> `D816V`, everything else -> `WT` (ChEMBL's target index for `CHEMBL1936` is wild-type KIT; there's no separate mutant target entry).

Aggregation is restricted to exact (`standard_relation == "="`) measurements for the point estimate — censored values (">10000 nM" etc.) are handled explicitly in the next step, not silently folded into a median here.

In [6]:
mutant_mask = df["assay_description"].str.contains("D816V", case=False, na=False)
df["kit_variant"] = np.where(mutant_mask, "D816V", "WT")

print(df["kit_variant"].value_counts())
print()
print("Unique compounds (ignoring variant):", df["molecule_chembl_id"].nunique())
print(
    "Unique (compound, variant) groups:",
    df.groupby(["molecule_chembl_id", "kit_variant"]).ngroups,
)

kit_variant
WT       6206
D816V    2291
Name: count, dtype: int64

Unique compounds (ignoring variant): 4724
Unique (compound, variant) groups: 5649


In [7]:
exact = df[(df["standard_relation"] == "=") & df["p_value"].notna()].copy()

agg = (
    exact.groupby(["molecule_chembl_id", "kit_variant"])
    .agg(
        canonical_smiles=("canonical_smiles", "first"),
        p_value_median=("p_value", "median"),
        p_value_std=("p_value", "std"),
        n_measurements=("p_value", "size"),
        n_documents=("document_chembl_id", "nunique"),
        standard_types=("standard_type", lambda s: ",".join(sorted(s.unique()))),
    )
    .reset_index()
)

print(f"Exact ('=') measurement rows: {len(exact)}")
print(f"Aggregated (compound, variant) rows: {len(agg)}")
print(f"  of which single-measurement groups: {(agg['n_measurements'] == 1).sum()}")
print(f"  of which multi-measurement groups: {(agg['n_measurements'] > 1).sum()}")
agg.sort_values("n_measurements", ascending=False).head()

Exact ('=') measurement rows: 6127
Aggregated (compound, variant) rows: 4093
  of which single-measurement groups: 2878
  of which multi-measurement groups: 1215


,molecule_chembl_id,kit_variant,canonical_smiles,p_value_median,p_value_std,n_measurements,n_documents,standard_types
4091,CHEMBL941,WT,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc...,6.966576,0.939419,53,31,"IC50,Kd,Ki"
1565,CHEMBL535,WT,CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C2\C(=O)Nc3ccc(F...,8.050610,1.294483,51,23,"IC50,Kd,Ki"
48,CHEMBL124660,WT,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc...,6.920819,1.359167,28,10,"IC50,Kd,Ki"
75,CHEMBL1336,WT,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,7.508638,0.609886,28,15,"IC50,Kd"
852,CHEMBL388978,WT,CN[C@@H]1C[C@H]2O[C@@](C)([C@@H]1OC)n1c3ccccc3...,7.721246,0.637231,26,17,"IC50,Kd"


## 4. Handle censored values

Design Doc §4.4: decide explicitly whether to drop, cap, or flag censored values (`>10000 nM`, `<1 nM`, etc.) rather than silently mishandling them.

**1,556 of the 5,649 `(compound, variant)` groups (28%) have no exact measurement.** Dropping all of them outright would remove over a quarter of the dataset and disproportionately strip out weak/inactive compounds — worsening exactly the ChEMBL activity bias Design Doc §9 already flags ("under-representation of negative/inactive results").

**Decision: cap, don't drop, wherever there's a usable one-directional bound; drop only the genuinely unusable remainder.**

- Groups that already have an exact measurement (§3): censored rows are ignored, the exact-median stands. No change here.
- Groups with only `>`/`>=` records (weak/inactive-only signal): capped at the **tightest bound**, i.e. the *minimum* `p_value` among those rows (the most restrictive claim about how weak the compound is — if a compound is both `>10000 nM` and `>1000 nM` in different assays, `>10000 nM` is the more informative constraint).
- Groups with only `<`/`<=` records (potent-only signal): capped at the *maximum* `p_value` among those rows (the tightest lower bound), by the same logic in reverse.
- Groups with **conflicting** `>` and `<` bounds and no exact value, or **no usable relation/value at all** (missing `standard_relation` and/or `standard_value`/`standard_units`): **dropped** — no reliable signal to assign a value from. This is a small fraction of the 1,556 no-exact groups (see the two "dropped" counts printed below), not the whole 28%.
- Every capped row is marked `censored=True` with a `censored_direction` (`>` or `<`), so it stays visibly distinct from exact measurements — Phase 4 modeling/evaluation can choose to exclude, downweight, or keep these as-is, rather than a downstream reader mistaking a capped bound for a precise label.

In [8]:
WEAK_RELATIONS = {">", ">="}
POTENT_RELATIONS = {"<", "<="}

exact_group_ids = set(agg[["molecule_chembl_id", "kit_variant"]].itertuples(index=False, name=None))
no_exact = df[
    ~df[["molecule_chembl_id", "kit_variant"]].apply(tuple, axis=1).isin(exact_group_ids)
]
n_no_exact_groups = no_exact.groupby(["molecule_chembl_id", "kit_variant"]).ngroups

censored = df[
    df["standard_relation"].isin(WEAK_RELATIONS | POTENT_RELATIONS) & df["p_value"].notna()
].copy()
censored_only = censored[
    ~censored[["molecule_chembl_id", "kit_variant"]]
    .apply(tuple, axis=1)
    .isin(exact_group_ids)
]
n_censored_only_groups = censored_only.groupby(["molecule_chembl_id", "kit_variant"]).ngroups

# Groups with no exact measurement AND no usable >/</>=/<= relation at all
# (missing standard_relation and/or standard_value/units) -- no signal to work with.
n_no_signal_groups = n_no_exact_groups - n_censored_only_groups


def cap_group(g):
    rel_set = set(g["standard_relation"])
    is_weak = rel_set <= WEAK_RELATIONS
    is_potent = rel_set <= POTENT_RELATIONS
    if is_weak:
        return pd.Series(
            {
                "canonical_smiles": g["canonical_smiles"].iloc[0],
                "p_value_median": g["p_value"].min(),  # tightest ">" bound
                "p_value_std": np.nan,
                "n_measurements": len(g),
                "n_documents": g["document_chembl_id"].nunique(),
                "standard_types": ",".join(sorted(g["standard_type"].unique())),
                "censored": True,
                "censored_direction": ">",
            }
        )
    if is_potent:
        return pd.Series(
            {
                "canonical_smiles": g["canonical_smiles"].iloc[0],
                "p_value_median": g["p_value"].max(),  # tightest "<" bound
                "p_value_std": np.nan,
                "n_measurements": len(g),
                "n_documents": g["document_chembl_id"].nunique(),
                "standard_types": ",".join(sorted(g["standard_type"].unique())),
                "censored": True,
                "censored_direction": "<",
            }
        )
    return None  # conflicting > and < bounds, no exact value -> ambiguous, dropped


capped = (
    censored_only.groupby(["molecule_chembl_id", "kit_variant"])
    .apply(cap_group, include_groups=False)
    .dropna(how="all")
    .reset_index()
)

n_dropped_ambiguous = n_censored_only_groups - len(capped)

print(f"Groups with no exact measurement: {n_no_exact_groups}")
print(f"  no usable relation at all (missing/unparseable) -> dropped: {n_no_signal_groups}")
print(f"  has a usable >/</>=/<=  bound: {n_censored_only_groups}")
print(f"    capped (kept):       {len(capped)}")
print(f"      weak (> bound):    {(capped['censored_direction'] == '>').sum()}")
print(f"      potent (< bound):  {(capped['censored_direction'] == '<').sum()}")
print(f"    dropped (conflicting > and < bounds): {n_dropped_ambiguous}")
print(
    f"\nTotal dropped for lack of usable signal: {n_no_signal_groups + n_dropped_ambiguous} "
    f"of {n_no_exact_groups} no-exact groups"
)

Groups with no exact measurement: 1556
  no usable relation at all (missing/unparseable) -> dropped: 79
  has a usable >/</>=/<=  bound: 1477
    capped (kept):       1476
      weak (> bound):    1065
      potent (< bound):  411
    dropped (conflicting > and < bounds): 1

Total dropped for lack of usable signal: 80 of 1556 no-exact groups


## 5. Combine exact + capped-censored rows, then canonicalize SMILES

First, combine the exact-median rows (§3) and the capped-censored rows (§4) into a single cleaned table — one row per `(molecule_chembl_id, kit_variant)`, `censored=False` for the exact ones.

Then canonicalize `canonical_smiles` via RDKit (Design Doc §5.1), to collapse any representation duplicates and catch unparseable structures before Phase 3 featurization.

In [9]:
agg["censored"] = False
agg["censored_direction"] = np.nan

df_clean = pd.concat([agg, capped], ignore_index=True)
df_clean = df_clean.rename(columns={"p_value_median": "p_value"})
# groupby-apply on `capped` can leave `censored` as object dtype (mixing Python bool
# objects in); force it back to bool so `~df_clean["censored"]` behaves as logical NOT,
# not Python's bitwise ~ (~True == -2, not False).
df_clean["censored"] = df_clean["censored"].astype(bool)

print(f"Combined cleaned table: {len(df_clean)} rows")
print(f"  exact:    {(~df_clean['censored']).sum()}")
print(f"  censored: {df_clean['censored'].sum()}")
df_clean.head()

Combined cleaned table: 5569 rows
  exact:    4093
  censored: 1476


,molecule_chembl_id,kit_variant,canonical_smiles,p_value,p_value_std,n_measurements,n_documents,standard_types,censored,censored_direction
0,CHEMBL101253,WT,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6.677781,1.308074,17.0,7.0,"IC50,Kd,Ki",False,NaN
1,CHEMBL101683,WT,O=C(Nc1ccc(Cl)cc1)c1ccccc1NCc1ccncc1,6.619789,NaN,1.0,1.0,IC50,False,NaN
2,CHEMBL102301,WT,COc1cc2ncnc(N3CCN(/C(S)=N\Cc4ccc5c(c4)OCO5)CC3...,6.201152,0.645482,2.0,2.0,IC50,False,NaN
3,CHEMBL102346,WT,COc1cc2ncnc(N3CCN(C(=O)Nc4ccc(Oc5ccccc5)cc4)CC...,7.301030,NaN,1.0,1.0,IC50,False,NaN
4,CHEMBL103667,WT,Cc1ccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(OCCN3CCO...,6.585027,0.462771,11.0,3.0,Kd,False,NaN


In [10]:
from rdkit import Chem


def rdkit_canonical_smiles(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


df_clean["rdkit_smiles"] = df_clean["canonical_smiles"].apply(rdkit_canonical_smiles)

n_invalid = df_clean["rdkit_smiles"].isna().sum()
n_diff = (
    (df_clean["rdkit_smiles"] != df_clean["canonical_smiles"]) & df_clean["rdkit_smiles"].notna()
).sum()
print(f"Rows with missing/unparseable SMILES: {n_invalid} / {len(df_clean)}")
print(f"Rows where RDKit-canonical differs from ChEMBL's canonical_smiles: {n_diff}")
df_clean.loc[df_clean["rdkit_smiles"].isna(), ["molecule_chembl_id", "kit_variant", "canonical_smiles"]]

Rows with missing/unparseable SMILES: 4 / 5569
Rows where RDKit-canonical differs from ChEMBL's canonical_smiles: 0


,molecule_chembl_id,kit_variant,canonical_smiles
2803,CHEMBL5899492,WT,None
2976,CHEMBL5920243,D816V,None
3880,CHEMBL6045764,WT,None
5385,CHEMBL5949948,D816V,NaN


In [11]:
n_before = len(df_clean)
df_clean = df_clean[df_clean["rdkit_smiles"].notna()].copy()
df_clean["canonical_smiles"] = df_clean["rdkit_smiles"]
df_clean = df_clean.drop(columns=["rdkit_smiles"])

print(f"Dropped {n_before - len(df_clean)} rows with unparseable/missing SMILES")
print(f"Remaining: {len(df_clean)} rows")

# Duplicate check: same canonical structure + same kit_variant under different
# molecule_chembl_id (e.g. salt vs. freebase ChEMBL entries) would need re-aggregation.
dup_groups = df_clean.groupby(["canonical_smiles", "kit_variant"])["molecule_chembl_id"].nunique()
n_dup_structures = (dup_groups > 1).sum()
print(f"Distinct structures (by RDKit-canonical SMILES) mapping to >1 molecule_chembl_id within the same variant: {n_dup_structures}")

Dropped 4 rows with unparseable/missing SMILES
Remaining: 5565 rows
Distinct structures (by RDKit-canonical SMILES) mapping to >1 molecule_chembl_id within the same variant: 0
